In [2]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

import re
import math
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import traceback

from datetime import datetime

import concurrent.futures
import os
import json
from datetime import datetime

In [3]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

# Load the datasets
FOLIO = load_json('../../testingDatasets/FOLIOsampled_train.json')
PROOF = load_json('../../testingDatasets/ProofWriterSampled_train.json')

# Load Examplars
hypothesis_CoT_prompt_examples = open('../../research/reasoning/prompt_examples/HFP_CoT.txt').read()
hypothesis_Standard_prompt_examples = open('../../research/reasoning/prompt_examples/HFP_Stan.txt').read()
hypothesis_CCoT_prompt_examples = open('../../research/reasoning/prompt_examples/HFP_Complex.txt').read()
CoT_prompt_examples = open('../../research/reasoning/prompt_examples/CoT.txt').read()
Standard_prompt_examples = open("../../research/reasoning/prompt_examples/HFP_Stan.txt").read()
CCoT_prompt_examples = open("../../research/reasoning/prompt_examples/ComplexCoT.txt").read()

In [4]:
endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
model_name = "gpt-4o"
deployment = "gpt-4o"
subscription_key = os.environ["AZURE_OPENAI_API_KEY"]
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff_4o(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=1512,
        temperature=0.0,
        model=deployment
    )

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
model_name = "gpt-35-turbo"
deployment = "gpt-35-turbo"
subscription_key = os.environ["AZURE_OPENAI_API_KEY"]
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff_35(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=1512,
        temperature=0.0,
        model=deployment
    )

In [5]:
def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return num
    except ValueError:
        return None

def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # Remove everything except digits, dot, minus
    try:
        return float(cleaned)
    except ValueError:
        return None

def answer_extractor(ans_model):
    # Pattern 1: Markdown-style "### Final Answer:" followed by a number in a sentence
    match = re.search(
        r'###\s*Final Answer:\s*.*?\**\$?(-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?)\**',
        ans_model,
        re.IGNORECASE | re.DOTALL
    )
    if match:
        value_str = match.group(1).replace(",", "")  # Remove commas
        return clean_and_truncate(value_str)
    # Pattern 2: Standard numerical answer formats
    match = re.search(
        r'(?:the answer is|final answer:)\s*\**\$?(-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?(?:e[+-]?\d+)?)\**',
        ans_model,
        re.IGNORECASE
    )
    if match:
        value_str = match.group(1).replace(",", "")  # Remove commas
        return clean_and_truncate(value_str)

    # Pattern 3: Multiple-choice answer (A-E)
    match = re.search(r'the answer is\s*\**([A-E])\**', ans_model, re.IGNORECASE)
    if not match:
        match = re.search(r'\b([A-E])\b', ans_model.strip()[-5:], re.IGNORECASE)

    if match:
        predicted_choice = match.group(1).upper()
        if predicted_choice in ['A', 'B', 'C', 'D', 'E']:
            return predicted_choice

    return None

def ground_truth_extractor(d, database_name):
    return d['correct']
    
def question_extractor(d, database_name):
    return f"{d['question']}\nOptions:\n{"\n".join(d['options'])}"


def generate_message(question, prompt_type, examples=None, dataset_name=None):
    if examples is None:
        examples = ""

    # Define dynamic answer suffix for logic datasets
    answer_suffix = "The answer is <True / False / Uncertain>" if dataset_name in ["FOLIO", "ProofWriter"] else "The answer is <value>"

    # === Prompt instruction by prompt_type ===
    if prompt_type == "HFP-CoT":
        prompt_instruction = (
            "First, write a high-level hypothesis about the logical relationships relevant to the question. "
            "Then, think step by step through these relationships to determine whether the statement is true, false, or uncertain."
        )

    elif prompt_type == "HFP-Standard":
        prompt_instruction = (
            "Start with a high-level hypothesis of what conclusion can be drawn from the context. "
            "Then, use that plan to directly evaluate whether the statement is true, false, or uncertain."
        )

    elif prompt_type == "HFP-Complex-CoT":
        prompt_instruction = (
            "Write a high-level hypothesis first. Then break the reasoning into labeled sub-steps that each analyze part of the context. "
            "Use those steps to decide whether the final statement is true, false, or uncertain."
        )

    elif prompt_type == "CoT":
        prompt_instruction = (
            "Think step by step through the relationships in the context to determine whether the statement is true, false, or uncertain."
        )

    elif prompt_type == "Standard":
        prompt_instruction = (
            "Read the context carefully and answer whether the statement is true, false, or uncertain."
        )

    elif prompt_type == "Complex-CoT":
        prompt_instruction = (
            "Break the reasoning process into labeled sub-steps, each justifying part of the logic. "
            "Then conclude whether the statement is true, false, or uncertain."
        )

    else:
        raise ValueError(f"Unknown prompt_type: {prompt_type}")

    # === Build full user prompt ===
    prompt_q = (
        examples.strip() +
        "\n\nQ: " + question.strip() +
        f"\nA: {prompt_instruction} Write your final answer as: {answer_suffix}"
    )

    # === Compose message ===
    messages = [
        {
            "role": "system",
            "content": f"You are solving logical reasoning tasks using the {prompt_type} reasoning style. "
                       f"Follow the instructions and provide a clearly reasoned answer. "
                       f"Write your final answer clearly at the end as: {answer_suffix}"
        },
        {
            "role": "user",
            "content": prompt_q
        }
    ]

    return messages




In [6]:
import os
import json
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# === Prompt Types ===
prompt_types = [
    "HFP-CoT", "HFP-Standard", "HFP-Complex-CoT",
    "CoT", "Standard", "Complex-CoT"
]

# === Dataset Mapping ===
dataset_map = {
    "FOLIO": FOLIO,
    "ProofWriter": PROOF,
}

# === Load Few-Shot Examples ===
prompt_examples_map = {
    "HFP-CoT": hypothesis_CoT_prompt_examples,
    "HFP-Standard": hypothesis_Standard_prompt_examples,
    "HFP-Complex-CoT": hypothesis_CCoT_prompt_examples,
    "CoT": CoT_prompt_examples,
    "Standard": Standard_prompt_examples,
    "Complex-CoT": CCoT_prompt_examples
}

# === Define Which Models to Test ===
models_to_run = {
    "gpt-3.5-turbo": completion_with_backoff_35,
    "gpt-4o": completion_with_backoff_4o,
}

# === Results Containers ===
all_results = {}
summary_results = {}

# === Process One Question ===
def process_item(item, dataset_name, prompt_type, few_shot_examples, completion_fn):
    try:
        question = question_extractor(item, dataset_name)
        ground_truth = ground_truth_extractor(item, dataset_name)
    except Exception as e:
        return {
            "question": "[ERROR extracting question]",
            "response": str(e),
            "parsed_answer": None,
            "is_correct": False,
            "ground_truth": None
        }

    messages = generate_message(
        question=question,
        prompt_type=prompt_type,
        examples=few_shot_examples,
    )

    try:
        model_response = completion_fn(messages)
        model_output = model_response.choices[0].message.content.strip()
    except Exception as e:
        model_output = f"[ERROR]: {str(e)}"

    parsed_answer = answer_extractor(model_output)

    is_correct = False
    try:
        if isinstance(parsed_answer, str) and isinstance(ground_truth, str):
            is_correct = parsed_answer.strip().upper() == ground_truth.strip().upper()
        elif isinstance(parsed_answer, (int, float)) and isinstance(ground_truth, (int, float, str)):
            is_correct = abs(float(parsed_answer) - float(ground_truth)) < 1e-3
    except:
        is_correct = False

    return {
        "question": question,
        "ground_truth": ground_truth,
        "response": model_output,
        "parsed_answer": parsed_answer,
        "is_correct": is_correct
    }

# === Main Loop: Model → Prompt → Dataset ===
for model_name, completion_fn in models_to_run.items():
    print(f"\n====== Running for MODEL: {model_name} ======")
    all_results[model_name] = {}
    summary_results[model_name] = {}

    for prompt_type in prompt_types:
        all_results[model_name][prompt_type] = {}
        summary_results[model_name][prompt_type] = {}

        few_shot_examples = prompt_examples_map[prompt_type]

        for dataset_name, dataset in dataset_map.items():
            print(f"→ Dataset: {dataset_name} | Prompt: {prompt_type}")
            results = []
            correct = 0
            total = 0

            def process_wrapper(item):
                return process_item(item, dataset_name, prompt_type, few_shot_examples, completion_fn)

            with ThreadPoolExecutor() as executor:
                futures = [executor.submit(process_wrapper, item) for item in dataset]
                for future in tqdm(as_completed(futures), total=len(dataset), desc=f"{model_name} - {prompt_type} - {dataset_name}"):
                    result = future.result()
                    results.append(result)
                    total += 1
                    if result["is_correct"]:
                        correct += 1

            accuracy = round((correct / total) * 100, 2) if total > 0 else 0.0
            print(f"  → Accuracy: {accuracy:.2f}% ({correct}/{total})")

            summary_results[model_name][prompt_type][dataset_name] = {
                "accuracy": accuracy,
                "correct": correct,
                "total": total
            }

            all_results[model_name][prompt_type][dataset_name] = results

            # Save to file
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            output_path = f"../../research/reasoning/logs/{model_name}_{prompt_type}_{dataset_name}_{timestamp}.json"
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(results, f, indent=4)

# === Final Summary ===
print("\n=== SUMMARY ACCURACY REPORT ===")
for model_name, prompts in summary_results.items():
    print(f"\nModel: {model_name}")
    for prompt_type, datasets in prompts.items():
        print(f"  Prompt: {prompt_type}")
        for dataset_name, stats in datasets.items():
            print(f"    {dataset_name:10s}: {stats['accuracy']}% ({stats['correct']}/{stats['total']})")



====== Running for MODEL: gpt-3.5-turbo ======
→ Dataset: FOLIO | Prompt: HFP-CoT


gpt-3.5-turbo - HFP-CoT - FOLIO: 100%|██████████| 205/205 [08:04<00:00,  2.36s/it]


  → Accuracy: 57.56% (118/205)
→ Dataset: ProofWriter | Prompt: HFP-CoT


gpt-3.5-turbo - HFP-CoT - ProofWriter: 100%|██████████| 205/205 [08:08<00:00,  2.38s/it]


  → Accuracy: 46.83% (96/205)
→ Dataset: FOLIO | Prompt: HFP-Standard


...

  → Accuracy: 33.17% (68/205)
→ Dataset: FOLIO | Prompt: Complex-CoT


gpt-4o - Complex-CoT - FOLIO: 100%|██████████| 205/205 [08:06<00:00,  2.37s/it]


  → Accuracy: 58.05% (119/205)
→ Dataset: ProofWriter | Prompt: Complex-CoT


gpt-4o - Complex-CoT - ProofWriter: 100%|██████████| 205/205 [08:02<00:00,  2.35s/it]

  → Accuracy: 47.32% (97/205)

=== SUMMARY ACCURACY REPORT ===

Model: gpt-3.5-turbo
  Prompt: HFP-CoT
    FOLIO     : 57.56% (118/205)
    ProofWriter: 46.83% (96/205)
  Prompt: HFP-Standard
    FOLIO     : 52.2% (107/205)
    ProofWriter: 45.37% (93/205)
  Prompt: HFP-Complex-CoT
    FOLIO     : 56.1% (115/205)
    ProofWriter: 49.27% (101/205)
  Prompt: CoT
    FOLIO     : 55.12% (113/205)
    ProofWriter: 44.88% (92/205)
  Prompt: Standard
    FOLIO     : 55.61% (114/205)
    ProofWriter: 36.59% (75/205)
  Prompt: Complex-CoT
    FOLIO     : 56.59% (116/205)
    ProofWriter: 48.29% (99/205)

Model: gpt-4o
  Prompt: HFP-CoT
    FOLIO     : 58.54% (120/205)
    ProofWriter: 41.46% (85/205)
  Prompt: HFP-Standard
    FOLIO     : 55.61% (114/205)
    ProofWriter: 48.78% (100/205)
  Prompt: HFP-Complex-CoT
    FOLIO     : 58.05% (119/205)
    ProofWriter: 49.76% (102/205)
  Prompt: CoT
    FOLIO     : 55.12% (113/205)
    ProofWriter: 44.88% (92/205)
  Prompt: Standard
    FOLIO     : 5